In [ ]:
import os
import scanpy as sc
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import anndata as ad
from pathlib import Path

# Enable vector-friendly mode (rasterizes large scatter plots automatically) 

sc.settings.vector_friendly = True 

# Increase DPI for high-resolution rendering 

sc.set_figure_params(dpi=300)

import matplotlib as mpl
from matplotlib.colors import Normalize
# Make PDF text editable in Illustrator
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

# Optional but helpful: use a common font
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["font.size"] = 7
mpl.rcParams["axes.titlesize"] = 7
mpl.rcParams["axes.labelsize"] = 7
mpl.rcParams["xtick.labelsize"] = 6
mpl.rcParams["ytick.labelsize"] = 6
mpl.rcParams["legend.fontsize"] = 6

In [ ]:
data_dir = Path("/home/anilprakash/labs/Mei/projects/anil/srda/notebooks/data/scrna_seq/kang/")
data_dir.mkdir(parents=True, exist_ok=True)

nhood_size = 50

#specify the condition column name in the adata.obs dataframe

condition_col = 'label'

celltype_col = 'cell_type'

#The first one should be disease state
conditions = ['stim', 'ctrl']

In [ ]:
adata = sc.read_h5ad(data_dir / f'adata_{nhood_size}_{conditions[0]}_{conditions[1]}.h5ad')

In [ ]:
adata.obs

In [ ]:
# `cluster_on_gene_scores` is provided by the sccst package (previously defined inline here).
from sccst.downstream import cluster_on_gene_scores

In [ ]:
adata = cluster_on_gene_scores(adata, n_top_genes=2000, resolutions=[0.1, 0.2, 0.3])

In [ ]:
sc.pl.umap(adata, color=[f'leiden_gene_score_{res}' for res in [0.1, 0.2, 0.3]], wspace=0.4)

In [ ]:
sc.pl.umap(adata, color='leiden_gene_score_0.3', wspace=0.4, legend_loc='on data')

In [ ]:
plt.figure(figsize=(8, 6))
sc.pl.umap(adata, color=celltype_col, wspace=0.4, title='Cell Type', show=False)
plt.savefig(data_dir / f'umap_cell_type.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
plt.figure(figsize=(8, 6))

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
# Reshape is needed for a single column
scores = adata.obs[['rra_cell_divergence_score']].values
adata.obs['cell_divergence_score_normalized'] = scaler.fit_transform(scores)

sc.pl.umap(adata, color='cell_divergence_score_normalized', cmap='viridis', alpha=0.6, title='Normalized Cell Divergence Scores', show=False)
plt.savefig(data_dir / f'umap_cell_divergence_score_{nhood_size}_{conditions[0]}_{conditions[1]}.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# `plot_ordered_violin` is provided by the sccst package (previously defined inline here).
from sccst.downstream import plot_ordered_violin

In [ ]:
plot_ordered_violin(adata, 'cell_divergence_score_normalized', celltype_col, method='mean', save_path=data_dir / f'violin_cell_divergence_score_normalized_{nhood_size}_{conditions[0]}_{conditions[1]}.pdf')

In [ ]:
# `analyze_global_drivers` is provided by the sccst package (previously defined inline here).
from sccst.downstream import analyze_global_drivers

In [ ]:
global_drivers_df = analyze_global_drivers(adata)

In [ ]:
# `plot_cell_type_specific_gene_score_markers` is provided by the sccst package (previously defined inline here).
from sccst.downstream import plot_cell_type_specific_gene_score_markers

In [ ]:
ct_marker_dict = plot_cell_type_specific_gene_score_markers(
    adata,
    cell_type_col=celltype_col,
    layer="rra_gene_score",
    is_dendrogram=False,
    top_k=3,
    vmin=-2,
    vmax=2,
    min_cells=5,
    save_path=data_dir / f'cell_type_specific_markers_matrixplot_{nhood_size}_{conditions[0]}_{conditions[1]}.pdf',
)

In [ ]:
# `plot_one_cell_type_specific_marker_gene_scores` is provided by the sccst package (previously defined inline here).
from sccst.downstream import plot_one_cell_type_specific_marker_gene_scores

In [ ]:
marker_genes, marker_stats_df =  plot_one_cell_type_specific_marker_gene_scores(
    adata,
    target_cell_type="NK cells",
    cell_type_col=celltype_col,
    layer="rra_gene_score",
    top_k=10,
    save_path=data_dir / f'cell_type_nk_cells_specific_markers_barplot_{nhood_size}_{conditions[0]}_{conditions[1]}.pdf'
)

In [ ]:
# `plot_single_celltype_lollipop` is provided by the sccst package (previously defined inline here).
from sccst.downstream import plot_single_celltype_lollipop

In [ ]:
# Example usage:
plot_single_celltype_lollipop(adata, cell_type='CD14+ Monocytes', cell_type_col=celltype_col, layer='rra_gene_score', top_k=20)

In [ ]:
# `run_go_celltype` is provided by the sccst package (previously defined inline here).
from sccst.downstream import run_go_celltype

In [ ]:
# 1. Run and save to CSV
go_results = run_go_celltype(adata, cell_type_col=celltype_col, output_csv=data_dir / f'{celltype_col}_go_results.csv')

In [ ]:
go_results

In [ ]:
# `plot_go_enrichment` is provided by the sccst package (previously defined inline here).
from sccst.downstream import plot_go_enrichment

In [ ]:
# 2. Plot everything
plot_go_enrichment(go_results, top_k=5, data_dir=data_dir)

In [ ]:
# `visualize_gene_stir` is provided by the sccst package (previously defined inline here).
from sccst.downstream import visualize_gene_stir

In [ ]:
# ==============================================================================
# USAGE EXAMPLES
# ==============================================================================
up_gene = global_drivers_df.nlargest(1, 'mean_score').index[0]
down_gene = global_drivers_df.nsmallest(1, 'mean_score').index[0]
# 1. Plot Gene Expression (Two plots: Normal vs Tumor)
visualize_gene_stir(adata, up_gene, condition_col=condition_col, layer=None)
visualize_gene_stir(adata, down_gene, condition_col=condition_col, layer=None)


In [ ]:
# 2. Plot Disease Z-Score (One plot: Red/Blue map of significance)
visualize_gene_stir(adata, up_gene, condition_col=condition_col, layer='rra_gene_score')
visualize_gene_stir(adata, down_gene, condition_col=condition_col, layer='rra_gene_score')

In [ ]:
visualize_gene_stir(adata, "GZMA", condition_col=condition_col, layer=None)

In [ ]:
visualize_gene_stir(adata, "GZMA", condition_col=condition_col, layer='rra_gene_score')